# Prodata CAD-Gym — Quickstart

**What this notebook shows:**
- Install the Prodata CAD gym in one cell
- Load the `BracketDesign-v0` environment
- Write a bracket design in CadQuery and see how it scores
- Understand the reward dimensions (structural / cost / geometry)

**No API key required** — the open-source verifier runs entirely locally.

---

## 1. Install dependencies

This takes ~2 minutes on Colab the first time. CadQuery ships prebuilt wheels so no conda needed.

In [ ]:
# Install all dependencies
!pip install -q cadquery trimesh gymnasium pydantic

# Install the Prodata package directly from GitHub
!pip install -q git+https://github.com/prodata-ai/ProdataGym.git

print("Installation complete.")

## 2. Import and create the environment

In [ ]:
import gymnasium as gym
import prodata.cad_gym  # registers BracketDesign-v0

env = gym.make("prodata/BracketDesign-v0")
print("Environment created:", env)

## 3. Look at a task

Each episode gives the agent a bracket design specification. Let's see what task 001 looks like.

In [ ]:
obs, info = env.reset(options={"task_id": "cad_bracket_001"})

print("Task ID  :", info["task_id"])
print()
print("Brief    :", obs["task_description"])
print()
print(f"Load     : {obs['load_kg'][0]:.0f} kg")
print(f"Extension: {obs['extension_mm'][0]:.0f} mm")
print(f"Budget   : ${obs['max_cost_usd'][0]:.0f}")

## 4. Write a bracket design

The agent's action is **Python code** using [CadQuery](https://cadquery.readthedocs.io/).  
The code must define a variable called `result` of type `cadquery.Workplane`.

Here's a simple L-bracket:

In [ ]:
design_v1 = """
import cadquery as cq

# Simple L-bracket: horizontal shelf + vertical back plate
shelf = cq.Workplane("XY").box(120, 80, 10)          # horizontal arm
back  = cq.Workplane("XZ").box(120, 10, 100)          # vertical mounting plate
back  = back.translate((0, -35, 55))                  # position above shelf

result = shelf.union(back)
"""

obs, reward, terminated, truncated, info = env.step(design_v1)

print(f"Overall score : {reward:.3f}  (0=fail, 1=perfect)")
print(f"Passed        : {info['success']}")
print()
print("Dimension scores:")
for dim, score in info["dimension_scores"].items():
    bar = "\u2588" * int(score * 20)
    print(f"  {dim:12s}: {score:.3f}  {bar}")
if info["warnings"]:
    print()
    print("Warnings:", info["warnings"])

## 5. Improve the design

The verifier checks:
- **Structural** — safety factor (yield strength / bending stress) and deflection
- **Cost** — estimated material + machining cost vs budget
- **Geometry** — does it fit in the allowed bounding box?

A common mistake: making the bracket too tall wastes material and blows the budget.  
A bracket that's too thin fails structurally.  

Let's try a more efficient ribbed design:

In [ ]:
# Reset for a fresh episode on the same task
env.reset(options={"task_id": "cad_bracket_001"})

design_v2 = """
import cadquery as cq

# Ribbed L-bracket — more material-efficient than solid block
# Back plate: 8 mm thick, 100 mm tall
back = (
    cq.Workplane("XY")
    .box(100, 8, 100)
)

# Horizontal shelf: 8 mm thick, 110 mm extension
shelf = (
    cq.Workplane("XZ")
    .box(100, 8, 110)
    .translate((0, 46, -55))
)

# Diagonal gusset for stiffness
gusset = (
    cq.Workplane("XY")
    .polyline([(0, 0), (0, 80), (60, 0)])
    .close()
    .extrude(8)
    .translate((-50, -4, -50))
)

result = back.union(shelf).union(gusset)
"""

obs, reward, terminated, truncated, info = env.step(design_v2)

print(f"Overall score : {reward:.3f}")
print(f"Passed        : {info['success']}")
print()
print("Dimension scores:")
for dim, score in info["dimension_scores"].items():
    bar = "\u2588" * int(score * 20)
    print(f"  {dim:12s}: {score:.3f}  {bar}")
if info["warnings"]:
    print("Warnings:", info["warnings"])

## 6. Browse all 50 tasks

The benchmark includes 50 tasks across three difficulty levels and 9 materials.

In [ ]:
import json
from pathlib import Path

# Load task metadata (no simulation needed)
tasks_path = Path(prodata.cad_gym.__file__).parent / "tasks" / "brackets_basic.json"
with open(tasks_path) as f:
    tasks = json.load(f)

print(f"Total tasks: {len(tasks)}\n")

# Group by difficulty
for level in ("easy", "medium", "hard"):
    group = [t for t in tasks if t["difficulty"] == level]
    print(f"{'='*60}")
    print(f"{level.upper()} ({len(group)} tasks)")
    print(f"{'='*60}")
    for t in group:
        req = t["requirements"]
        print(
            f"  {t['task_id']}  "
            f"{req['load_kg']:>5.0f}kg  "
            f"{req['extension_mm']:>4.0f}mm  "
            f"{req['material']:<20s}  "
            f"${req['max_cost_usd']:>5.0f}  "
            f"{t['description'][:60]}..."
        )
    print()

## 7. Sketch of an RL training loop

This shows the structure of how you'd plug in your own language model agent.  
Replace `your_model.generate(prompt)` with your actual model call.

In [ ]:
def build_prompt(obs: dict) -> str:
    return (
        f"Design a mechanical bracket using CadQuery Python.\n\n"
        f"Task: {obs['task_description']}\n\n"
        f"Requirements:\n"
        f"  Load:      {obs['load_kg'][0]:.0f} kg\n"
        f"  Extension: {obs['extension_mm'][0]:.0f} mm from mounting surface\n"
        f"  Budget:    ${obs['max_cost_usd'][0]:.0f}\n\n"
        f"Your code must define a variable `result` of type cadquery.Workplane.\n"
        f"Import cadquery as cq at the top."
    )


# --- Plug in your model here ---
def your_model_generate(prompt: str) -> str:
    # Replace this with: openai.chat(), anthropic.messages.create(), etc.
    raise NotImplementedError("Replace with your model call")
# --------------------------------


# Training loop skeleton
N_EPISODES = 100
rewards = []

for episode in range(N_EPISODES):
    obs, info = env.reset()          # random task each episode
    prompt = build_prompt(obs)

    code = your_model_generate(prompt)   # <-- your model generates CadQuery code

    _, reward, _, _, step_info = env.step(code)
    rewards.append(reward)

    # Use reward as training signal for your RL algorithm (PPO, REINFORCE, etc.)
    # your_rl_algorithm.update(prompt, code, reward)

    if episode % 10 == 0:
        avg = sum(rewards[-10:]) / min(len(rewards), 10)
        print(f"Episode {episode:>4d} | reward={reward:.3f} | avg10={avg:.3f} | task={info['task_id']}")

print(f"\nFinal average reward: {sum(rewards)/len(rewards):.3f}")
env.close()

## Next steps

- **Pro verifier** — the basic verifier here is intentionally simple and can be gamed. The Pro API adds multi-dimensional scoring and anti-gaming detection. Sign up at [prodata.ai](https://prodata.ai).
- **More domains** — Solar/PV and RF engineering environments coming soon.
- **Leaderboard** — submit your model's pass rate: open a PR to [github.com/prodata-ai/ProdataGym](https://github.com/prodata-ai/ProdataGym).
- **Questions / bugs** — open an issue on GitHub.